# I3.5-PQ: Pinned-Policy Counterfactual Action Values

This notebook collects causal action data under the **pinned Qwen2.5-7B-Instruct Q4_K_M** downstream policy.

For each of the 1056 interventions in the frozen schedule:
1. Restore the checkpoint
2. Force the specified action
3. For terminal actions (ANSWER, DEFER, STOP): record outcome immediately
4. For non-terminal actions: return control to the pinned Qwen policy
5. Record Q^{pi_Qwen}(s,a) = realized utility under pinned downstream policy

Binding: `I3_5_PINNED_POLICY_V1`

## 1. Environment Setup

In [ ]:
# Install llama-cpp-python with GPU support
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python==0.3.35 --no-cache-dir

In [ ]:
# Clone the repository
!git clone -b i3.12-semantic-relation-ablation https://github.com/dawsonblock/Daph-ex-research-gate-c2-beir-retrieval.git /content/daph
%cd /content/daph

In [ ]:
# Download the Qwen2.5-7B-Instruct Q4_K_M GGUF model
# Verify SHA256: 65b8fcd92af6b4fefa935c625d1ac27ea29dcb6ee14589c55a8f115ceaaa1423
!wget -q https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-GGUF/resolve/main/qwen2.5-7b-instruct-q4_k_m.gguf -O /content/Qwen2.5-7B-Instruct-Q4_K_M.gguf
!sha256sum /content/Qwen2.5-7B-Instruct-Q4_K_M.gguf

In [ ]:
# Install other dependencies
!pip install scikit-learn numpy --quiet

## 2. Verify Binding and Checkpoints

In [ ]:
import json, sys, hashlib
sys.path.insert(0, '.')
sys.path.insert(0, 'scripts')

# Verify binding
with open('experiments/i3_5/pinned_policy/PINNED_POLICY_BINDING.json') as f:
    binding = json.load(f)
print(f"Binding: {binding['binding_id']}")
print(f"Model: {binding['model']['model_name']}")
print(f"GGUF SHA: {binding['model']['gguf_sha256'][:16]}...")
print(f"Temperature: {binding['generation_config']['temperature']}")

# Verify GGUF SHA
with open('/content/Qwen2.5-7B-Instruct-Q4_K_M.gguf', 'rb') as f:
    gguf_sha = hashlib.sha256(f.read()).hexdigest()
expected = binding['model']['gguf_sha256']
assert gguf_sha == expected, f"GGUF SHA mismatch: {gguf_sha[:16]}... != {expected[:16]}..."
print(f"GGUF SHA verified: {gguf_sha[:16]}...")

# Verify checkpoints
with open('experiments/i3_5/datasets/checkpoints_v1.jsonl') as f:
    n_checkpoints = sum(1 for _ in f)
print(f"Checkpoints: {n_checkpoints}")
assert n_checkpoints == 220, f"Expected 220 checkpoints, got {n_checkpoints}"

## 3. Run Pinned-Policy Causal Collection

In [ ]:
# Run the collection script
# This will take several hours on a T4 GPU (1056 interventions, each with LLM rollout)
!PYTHONPATH=. python3 scripts/collect_i3_5_pinned_policy_causal.py \
    --gguf-path /content/Qwen2.5-7B-Instruct-Q4_K_M.gguf \
    --output-dir experiments/i3_5/pinned_policy \
    --n-ctx 4096 \
    --max-rollout-steps 8 \
    --resume

## 4. Quick Results Summary

In [ ]:
import json
from pathlib import Path

# Load manifest
manifest_path = Path('experiments/i3_5/pinned_policy/pinned_causal_manifest_v1.json')
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print("=== Pinned-Policy Collection Manifest ===")
    for k, v in manifest.items():
        print(f"  {k}: {v}")
else:
    print("Manifest not found. Collection may not have completed.")

# Quick stats by action
from collections import defaultdict
by_action = defaultdict(list)
with open('experiments/i3_5/pinned_policy/pinned_causal_actions_v1.jsonl') as f:
    for line in f:
        r = json.loads(line)
        by_action[r['forced_action']].append(r)

print("\n=== Per-Action Summary ===")
for action, records in sorted(by_action.items()):
    n = len(records)
    utilities = [r['pinned_policy_utility'] for r in records]
    successes = sum(1 for r in records if r['pinned_policy_success'])
    mean_u = sum(utilities) / n if n else 0
    print(f"  {action:15s}: n={n:4d} mean_U={mean_u:8.2f} success_rate={successes/n:.4f}")

## 5. Train Model Ladder on Pinned-Policy Q Values

In [ ]:
# Train B0, B1, Linear, GBT (Q_CAUSAL_POLICY) on pinned-policy Q values
# Evaluate regret, top-1, top-2, subtype consistency
!PYTHONPATH=. python3 scripts/train_i3_5_pinned_model_ladder.py

## 6. Download Results (Optional)

In [ ]:
# Zip results for download
!cd experiments/i3_5/pinned_policy && zip -r /content/i3_5_pinned_policy_results.zip . && cd /content
print("Results zipped to /content/i3_5_pinned_policy_results.zip")
print("Download this file from the Colab file browser.")